<style>
.topic-header { background: linear-gradient(135deg, #e8f4f8 0%, #d4e8f0 100%); border-left: 4px solid #5ba4c9; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 10px 0; font-size: 15px; color: #1a3a4a; }
.concept-box { background: #eef6fa; border: 1px solid #c4dce8; border-radius: 8px; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #2a4a5a; }
.try-it { background: #fef9e7; border: 1px solid #f0d87a; border-radius: 8px; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #5a4a1a; }
.takeaway { background: #e8f5e8; border: 1px solid #a8d5a8; border-radius: 8px; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #2a5a2a; }
.warning-box { background: #fdf0f0; border: 1px solid #e8b0b0; border-radius: 8px; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #6a2a2a; }
.where-box { background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 0 8px 8px 0; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #4a3000; }
.fix-box { background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 0 8px 8px 0; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #1b5e20; }
.diagram-box { background: #f8f9fa; border: 2px solid #dee2e6; border-radius: 12px; padding: 24px; margin: 16px 0; text-align: center; }
.compare-table { width: 100%; border-collapse: collapse; margin: 12px 0; }
.compare-table th { background: #d4e8f0; color: #1a3a4a; padding: 10px 14px; text-align: left; border: 1px solid #c4dce8; }
.compare-table td { padding: 10px 14px; border: 1px solid #dee2e6; font-size: 13px; }
.compare-table tr:nth-child(even) { background: #f8fbfd; }
.section-divider { border: none; border-top: 2px solid #d4e8f0; margin: 25px 0; }
</style>

<div class='topic-header'>
<h1>E08 &middot; Day 3 &middot; RAG End-to-End &mdash; Ground the Model in YOUR Documents</h1>
<p><strong>GenAI for Engineering Managers &bull; Exercise 8 of 15 &bull; Opens Day 3</strong> &nbsp;|&nbsp; From an assistant that chats fluently to one that answers from your handbook and runbooks &mdash; with sources</p>
</div>

**Why this matters at your altitude.** Every "AI assistant for our docs" pitch your teams will bring you &mdash; support copilots, onboarding bots, runbook assistants &mdash; is, underneath, the pattern in this notebook: **Retrieval-Augmented Generation (RAG)**. In the next hour you will watch a raw model fail on questions about our own engineering handbook, then watch the *same model* answer them correctly &mdash; with citations &mdash; after we give it a retrieval layer. No fine-tuning, no training run, no data leaving the room. When a vendor quotes months for "training the AI on your documents", this session is your calibration for what that sentence should actually mean, cost, and take.

<hr class='section-divider'>

## Part 1 &mdash; Closing the E07 Gap: The Assistant That Knows Nothing About Us

E07 ended with a working, secure, shareable chat assistant &mdash; and one honest admission:
**everything it "knows" is a handful of hand-pasted knowledge-base entries inside its prompt.**
It has never seen our engineering handbook, our on-call policy, or our store-systems runbooks.
And E07's debug panel showed why we cannot simply paste them all in: every token of that prompt
rides along on **every single turn**, and real document sets run to thousands of pages.

Today we close that gap. Instead of stuffing documents into the prompt, we build a pipeline that
**fetches only the relevant passages at question time** and hands just those to the model.

<div class='concept-box'>
<strong>The mental model: an open-book exam with a librarian</strong><br><br>
A raw LLM answers from memory &mdash; a <em>closed-book exam</em> over the public internet as of its training date.
RAG turns this into an <em>open-book exam</em>: before the model answers, a retrieval step finds the most relevant
pages of <strong>your</strong> documents and places them in front of the model, with instructions to answer
<em>only</em> from what it was handed.<br><br>
Three moving parts, and only three:
<ol>
<li><strong>Retrieve</strong> &mdash; find the document passages most relevant to the question</li>
<li><strong>Augment</strong> &mdash; insert those passages into the prompt as context</li>
<li><strong>Generate</strong> &mdash; the LLM answers, grounded in that context</li>
</ol>
Everything else in this notebook &mdash; chunking, embeddings, vector stores &mdash; exists only to make step 1 fast and accurate.
</div>

<div class='diagram-box'>
<p style='font-size:13px; color:#888; margin-bottom:12px;'>The RAG pipeline we build in this session</p>
<table style='width:100%; border-collapse:collapse; font-size:13px;'>
<tr>
<td style='padding:10px; text-align:center; background:#eef6fa; border-radius:8px;'>&#128196;<br><strong>Your documents</strong><br>handbook + runbook</td>
<td style='padding:6px;'>&rarr;</td>
<td style='padding:10px; text-align:center; background:#eef6fa; border-radius:8px;'>&#9986;&#65039;<br><strong>Chunk</strong><br>split into passages</td>
<td style='padding:6px;'>&rarr;</td>
<td style='padding:10px; text-align:center; background:#eef6fa; border-radius:8px;'>&#128290;<br><strong>Embed</strong><br>text &rarr; vectors</td>
<td style='padding:6px;'>&rarr;</td>
<td style='padding:10px; text-align:center; background:#eef6fa; border-radius:8px;'>&#128451;&#65039;<br><strong>Vector store</strong><br>searchable index</td>
</tr>
</table>
<p style='margin:10px 0 4px 0;'>&#8595; at question time &#8595;</p>
<table style='width:100%; border-collapse:collapse; font-size:13px;'>
<tr>
<td style='padding:10px; text-align:center; background:#fff3e0; border-radius:8px;'>&#10067;<br><strong>Question</strong></td>
<td style='padding:6px;'>&rarr;</td>
<td style='padding:10px; text-align:center; background:#fff3e0; border-radius:8px;'>&#128269;<br><strong>Retrieve</strong><br>top-k similar chunks</td>
<td style='padding:6px;'>&rarr;</td>
<td style='padding:10px; text-align:center; background:#fff3e0; border-radius:8px;'>&#128221;<br><strong>Augment</strong><br>chunks + question &rarr; prompt</td>
<td style='padding:6px;'>&rarr;</td>
<td style='padding:10px; text-align:center; background:#e8f5e8; border-radius:8px;'>&#129302;<br><strong>Generate</strong><br>grounded answer + sources</td>
</tr>
</table>
</div>

<hr class='section-divider'>

## Part 2 &mdash; Setup

<div class='concept-box'>
Three model-facing pieces and three document loaders. The <strong>generation model</strong> writes the
answers, the <strong>embedding model</strong> turns text into vectors, and <strong>FAISS</strong> is the
index we search. The loaders let us read a web page, a Word file, a PDF and a text file with the
<em>same</em> downstream pipeline &mdash; that uniformity is the whole point.
</div>


In [ ]:
%pip install -qU langchain langchain-openai langchain-community langchain-text-splitters faiss-cpu beautifulsoup4 pypdf docx2txt

In [ ]:
import os

# ── Configuration ────────────────────────────────────────────────────
# Facilitator note: the key below is set live during the session.
os.environ['OPENAI_API_KEY'] = 'PASTE_THE_KEY_SHARED_IN_SESSION_HERE'

# WebBaseLoader asks for a user agent; set one so the fetch is polite and reproducible.
os.environ['USER_AGENT'] = 'Mozilla/5.0 (GenAI-training-notebook)'

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import (
    WebBaseLoader,      # web pages
    TextLoader,         # .txt
    Docx2txtLoader,     # .docx
    PyPDFLoader,        # .pdf
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

llm       = ChatOpenAI(model='gpt-4.1-nano', temperature=0)      # generation — small, cheap, fast
embedding = OpenAIEmbeddings(model='text-embedding-3-small')     # embedding  — text -> 1536-dim vectors

print('LLM and embedding model ready.')
print('Generation: gpt-4.1-nano  |  Embeddings: text-embedding-3-small')

<hr class='section-divider'>

## Part 3 &mdash; The Baseline: Watch a Confident Model Be Wrong

<div class='concept-box'>
Before we build anything, we need to <em>see the problem</em>. We ask the raw model a question about a
public web page &mdash; the Wikipedia entry for the Indian Space Research Organisation. Nothing secret,
nothing internal. The model has almost certainly read about ISRO during training.<br><br>
The catch is in <em>how</em> we ask. We demand <strong>an exact figure and a named person</strong>.
Vague questions get hedged answers; questions demanding specifics get confident inventions.
</div>

<div class='warning-box'>
<strong>Watch for this, because it is the entire lesson:</strong> the model will not say
"I am not sure." It will answer in the voice of someone reading the page. Note the phrasing it
uses &mdash; <em>"the page states..."</em> &mdash; when it has never seen the page.
</div>


In [ ]:
# A question that demands specifics: an exact budget figure and a named office-holder.
# The model has read *about* ISRO in training, but it has not read *this page*, and
# facts like "who currently chairs the organisation" drift over time.

baseline_question = (
    "According to the Wikipedia page on the Indian Space Research Organisation, "
    "what is the exact annual budget figure mentioned for ISRO, and who is listed "
    "as the current chairperson? Give me the response in a single line."
)

baseline_answer = llm.invoke(baseline_question).content

print("QUESTION:")
print(baseline_question)
print("\nBASELINE MODEL (no RAG):")
print(baseline_answer)

In [ ]:
# A second one, even more specific — an exact headcount.

baseline_question_2 = "What is the exact number of employees at ISRO? Answer in one line."
baseline_answer_2 = llm.invoke(baseline_question_2).content

print("QUESTION:")
print(baseline_question_2)
print("\nBASELINE MODEL (no RAG):")
print(baseline_answer_2)

<div class='try-it'>
<strong>What to observe &mdash; ask the room before scrolling on:</strong>
<ul style='margin:6px 0 0 18px;'>
<li>Did the model hedge, or did it state a figure as fact?</li>
<li>It said <em>"according to the page"</em>. Which page? It never opened one.</li>
<li>If you did not already know the right answer, <strong>would anything here look wrong to you?</strong></li>
</ul>
That last question is the one to sit on. A hallucination is not a garbled answer &mdash;
it is a <em>fluent, plausible, well-formatted</em> answer that happens to be false.
</div>


<hr class='section-divider'>

## Part 4 &mdash; RAG Over a URL, Step by Step

<div class='concept-box'>
We now hand the model the page. Four steps, and we will look at the output of every one:
<strong>load &rarr; chunk &rarr; embed &rarr; retrieve</strong>. Nothing here is magic, and by the end
you will be able to point at the exact step where a production RAG system usually goes wrong.
</div>


### Step 4.1 &mdash; Load the page

<div class='concept-box'>
<code>WebBaseLoader</code> fetches the URL and strips the HTML down to text. Note the character count:
this single page is far too large to paste into a prompt on every turn &mdash; which is precisely
why the remaining steps exist.
</div>


In [ ]:
url = "https://en.wikipedia.org/wiki/Indian_Space_Research_Organisation"

loader = WebBaseLoader(web_paths=[url])
documents = loader.load()

print(f"Loaded {len(documents)} document(s) from: {url}")
print(f"Total characters: {len(documents[0].page_content):,}")
print("\nFirst 700 characters of the loaded content:")
print("-" * 60)
print(documents[0].page_content[:700])

<div class='try-it'>
<strong>What to observe:</strong> that character count. Roughly 145,000 characters is on the order of
36,000 tokens &mdash; and it would ride along on <em>every single turn</em> of a conversation.
One page. Now imagine the handbook, the runbooks and five years of policy.
</div>


### Step 4.2 &mdash; Chunk the document

<div class='concept-box'>
<strong>Why chunk at all?</strong> Two reasons, and both matter in production.<br><br>
<strong>1. Precision.</strong> An embedding squeezes a whole passage into one vector. Embed 145,000
characters as a single vector and the meaning is averaged into mush &mdash; it matches everything
and nothing.<br><br>
<strong>2. Budget.</strong> We send only the matching chunks to the model, not the document.<br><br>
<strong>And why overlap?</strong> A hard cut can slice a sentence &mdash; or a fact &mdash; in half.
The 100-character overlap means a fact sitting on a boundary still appears whole in one of the two chunks.
</div>


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # target characters per chunk
    chunk_overlap=100,    # characters repeated between neighbours, so facts aren't cut in half
)

chunks = text_splitter.split_documents(documents)

print(f"Original document : {len(documents[0].page_content):,} characters")
print(f"After splitting   : {len(chunks)} chunks")
print(f"\nSample chunk (index 5):")
print("-" * 60)
print(chunks[5].page_content)
print("-" * 60)
print(f"Chunk length: {len(chunks[5].page_content)} characters")

### Step 4.3 &mdash; Embed the chunks and build the vector store

<div class='concept-box'>
An <strong>embedding</strong> turns a chunk into a list of 1,536 numbers positioned so that similar
meanings sit near each other in space. "How fast must a SEV-1 be acknowledged?" and "acknowledgement
window for critical pages" contain almost no shared words, but land in nearly the same place.<br><br>
<strong>That is the leap from keyword search to meaning search</strong> &mdash; and it is why RAG
answers questions phrased in words that appear nowhere in the source.<br><br>
<strong>FAISS</strong> is the index holding those vectors and finding the nearest ones fast. This cell
is where the API calls happen: every chunk gets embedded.
</div>


In [ ]:
vectorstore = FAISS.from_documents(chunks, embedding)
retriever   = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

print(f"Vector store built: {len(chunks)} chunks indexed.")
print("Each chunk is now a 1,536-dimension vector.")
print("The retriever will return the top 4 most similar chunks per question.\n")

# Peek behind the curtain: retrieval ALONE, with no model involved yet.
preview = retriever.invoke("Who leads ISRO and what is its budget?")
print("Top retrieved chunk for 'Who leads ISRO and what is its budget?':")
print("-" * 60)
print(preview[0].page_content[:500])

<div class='try-it'>
<strong>What to observe:</strong> no model has been asked anything yet. That was <em>pure retrieval</em>
&mdash; the question became a vector, and we pulled back the nearest chunks. If the right passage does
not appear here, no amount of prompt engineering downstream will save the answer.
<br><br>
<strong>Manager's version of that sentence:</strong> when a RAG system gives a bad answer, check
retrieval before you blame the model. Most of the time the model never received the fact.
</div>


### Step 4.4 &mdash; The RAG chain: retrieve, augment, generate

<div class='concept-box'>
The prompt does the real governance work here. It orders the model to answer <strong>only</strong> from
the supplied context, and gives it an explicit, exact sentence to use when the context does not contain
the answer. Without that escape hatch, a helpful model fills the gap with invention.
</div>


In [ ]:
rag_prompt = PromptTemplate(
    template=(
        "You are a research assistant. Answer the question using ONLY the context provided below. "
        "Read the context carefully — the answer may be phrased differently from the question. "
        "If, after careful reading, the context truly contains nothing that answers the question, "
        "reply exactly: 'The provided context does not contain information to answer this question.'"
        "\n\n"
        "Context:\n{context}\n\n"
        "Question: {question}\n\n"
        "Answer:"
    ),
    input_variables=["context", "question"],
)

def ask_rag(question: str) -> str:
    """Full RAG pipeline: retrieve relevant chunks, augment the prompt, generate."""
    # 1. RETRIEVE — the question becomes a vector; nearest chunks come back
    relevant_chunks = retriever.invoke(question)

    # 2. AUGMENT — stitch those chunks into one context block
    context = "\n\n".join(chunk.page_content for chunk in relevant_chunks)

    # 3. GENERATE — the model sees the question AND the evidence
    filled = rag_prompt.format(context=context, question=question)
    return llm.invoke(filled).content

print("RAG chain ready: retrieve -> augment -> generate")

<hr class='section-divider'>

## Part 5 &mdash; The Rematch: Same Question, Grounded Model

<div class='concept-box'>
Identical question, identical model, identical temperature. The <strong>only</strong> thing that changed
is that the model was handed four relevant chunks. Watch both answers side by side.
</div>


In [ ]:
for q, before in [(baseline_question, baseline_answer), (baseline_question_2, baseline_answer_2)]:
    print("=" * 72)
    print("QUESTION:", q)
    print("\n  BEFORE (no RAG, from memory):")
    print("   ", before.strip())
    print("\n  AFTER  (grounded in the page):")
    print("   ", ask_rag(q).strip())
    print()

<div class='takeaway'>
<strong>The takeaway to say out loud:</strong> we did not retrain the model, fine-tune it, or upgrade it.
We changed <em>what it could see at question time</em>. That is the entire mechanism &mdash; and it is
why RAG is usually the cheapest large improvement available to a team shipping an LLM feature.
</div>


<hr class='section-divider'>

## Part 6 &mdash; Two Tests Every RAG System Must Pass

<div class='concept-box'>
A grounded answer is only half the requirement. A RAG system has <strong>two</strong> jobs, and teams
routinely test only the first:<br><br>
<strong>Test A &mdash; does it find the right information?</strong> (retrieval accuracy)<br>
<strong>Test B &mdash; does it refuse when the answer is not there?</strong> (grounding discipline)<br><br>
A system that aces A and fails B is <em>more</em> dangerous than no system at all, because it has
earned the user's trust before it lies to them.
</div>


### Test A &mdash; retrieval accuracy

In [ ]:
test_questions = [
    "What are ISRO's major satellite launch vehicles?",
    "When was ISRO established and under which department does it operate?",
    "What are some notable missions conducted by ISRO?",
    "What is ISRO's human spaceflight programme?",
]

print("TEST A: can it fetch the right information?")
print("=" * 72)
for q in test_questions:
    print(f"\nQ: {q}")
    print(f"A: {ask_rag(q)}")
    print("-" * 72)

### Test B.1 &mdash; completely unrelated questions

<div class='concept-box'>
The easy half of the refusal test. Nothing about these questions is in the context, and the
gap is obvious. Expected behaviour: the exact refusal sentence, every time.
</div>


In [ ]:
unrelated_questions = [
    "What is the recipe for chicken biryani?",
    "Explain the balance sheet structure under Indian Accounting Standards.",
    "How does the GST input tax credit mechanism work?",
]

print("TEST B.1: unrelated questions — refusal expected")
print("=" * 72)
for q in unrelated_questions:
    print(f"\nQ: {q}")
    print(f"A: {ask_rag(q)}")
    print("-" * 72)

### Test B.2 &mdash; trick questions: plausible, in-domain, and absent

<div class='concept-box'>
<strong>This is the test that matters.</strong> These questions sound exactly like the source material.
They are about space, about ISRO, in the right register &mdash; and the answers are not on the page.
Retrieval will confidently return the four <em>nearest</em> chunks regardless, because nearest is not
the same as <em>relevant</em>. The model must notice that the retrieved text does not actually answer
the question and refuse anyway.
</div>


In [ ]:
trick_questions = [
    "What is NASA's Artemis programme timeline for lunar missions?",
    "What monthly salary does the ISRO chairman earn?",
    "How many ISRO employees are based specifically in Ahmedabad?",
]

print("TEST B.2: in-domain trick questions — no fabrication allowed")
print("=" * 72)
for q in trick_questions:
    print(f"\nQ: {q}")
    print(f"A: {ask_rag(q)}")
    print("-" * 72)

<div class='try-it'>
<strong>Ask the room:</strong> if one of these came back with a confident invented number, would your
current release process have caught it before it reached a customer? That question &mdash; not the
model's benchmark score &mdash; is what determines whether a RAG feature is safe to ship.
</div>


<hr class='section-divider'>

## Part 7 &mdash; Now Our Own Documents: Text, Word and PDF in One Index

<div class='concept-box'>
A web page was the easy case &mdash; public, clean, one format. Real enterprise knowledge is
<strong>scattered across formats</strong>: a handbook in plain text, a runbook someone maintains in Word,
a policy that only exists as a PDF.<br><br>
The pipeline does not change. Only the <strong>loader</strong> changes. Each loader's job is identical:
turn a file into text plus metadata, and hand it to the same splitter.
</div>

<table class='compare-table'>
<tr><th>Format</th><th>Loader</th><th>Our file</th><th>Contents</th></tr>
<tr><td><code>.txt</code></td><td><code>TextLoader</code></td><td>engineering_handbook.txt</td><td>On-call policy, change freeze, onboarding</td></tr>
<tr><td><code>.docx</code></td><td><code>Docx2txtLoader</code></td><td>store_systems_runbook.docx</td><td>RB-101 / RB-102, alert threshold table</td></tr>
<tr><td><code>.pdf</code></td><td><code>PyPDFLoader</code></td><td>supplier_returns_policy.pdf</td><td>Returns window, chargebacks, suspension</td></tr>
</table>


### Step 7.1 &mdash; Build the document set

<div class='concept-box'>
One script creates the Word and PDF files so the notebook is self-contained &mdash; nobody has to hunt
for an attachment. Run it once.
</div>


In [ ]:
# Creates store_systems_runbook.docx, supplier_returns_policy.pdf and shipment_status_memo.txt
%run ../data/setup_e08_docs.py

### Step 7.2 &mdash; Three loaders, one document list

In [ ]:
txt_docs  = TextLoader("../data/engineering_handbook.txt", encoding="utf-8").load()
docx_docs = Docx2txtLoader("../data/store_systems_runbook.docx").load()
pdf_docs  = PyPDFLoader("../data/supplier_returns_policy.pdf").load()

for label, docs in [("TXT ", txt_docs), ("DOCX", docx_docs), ("PDF ", pdf_docs)]:
    chars = sum(len(d.page_content) for d in docs)
    print(f"{label} | {len(docs):>2} document object(s) | {chars:>7,} characters | source: {docs[0].metadata.get('source')}")

all_docs = txt_docs + docx_docs + pdf_docs
print(f"\nCombined: {len(all_docs)} document objects from 3 different file formats.")

<div class='try-it'>
<strong>What to observe &mdash; the metadata.</strong> Every loader stamps a <code>source</code> onto
what it returns, and the PDF loader also stamps a <strong>page number</strong>. That metadata survives
chunking, which is what makes citation possible in Step 7.5. If you strip metadata at ingest, you can
never add citations later &mdash; a genuinely expensive mistake to discover in month six.
</div>


### Step 7.3 &mdash; Chunk and index the mixed corpus

<div class='concept-box'>
Identical splitter, identical FAISS call. The pipeline genuinely does not care that these chunks came
from three different file formats &mdash; by this point everything is just text and metadata.
</div>


In [ ]:
doc_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
doc_chunks   = doc_splitter.split_documents(all_docs)

doc_vectorstore = FAISS.from_documents(doc_chunks, embedding)
doc_retriever   = doc_vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

print(f"{len(all_docs)} documents -> {len(doc_chunks)} chunks -> indexed in FAISS")

# How many chunks came from each source?
from collections import Counter
counts = Counter(os.path.basename(c.metadata.get("source", "?")) for c in doc_chunks)
print("\nChunks per source file:")
for src, n in counts.items():
    print(f"  {src:32} {n:>3} chunks")

### Step 7.4 &mdash; Baseline again: the model has never seen any of this

<div class='concept-box'>
The ISRO page was <em>public</em> &mdash; the model failed on recency and specificity. These documents are
<em>private</em>. The model could not know them at any level of effort. <strong>Same failure, different
cause</strong> &mdash; and RAG is the same fix for both. Worth naming explicitly, because the room will
otherwise leave thinking RAG is only for fresh news.
</div>


In [ ]:
internal_question = (
    "In our company's engineering handbook, what is the acknowledgement window for the "
    "primary on-call when a SEV-1 page fires, and on which exact dates does our "
    "peak-season change freeze start and end? Answer with the specific numbers and dates."
)

print("QUESTION:", internal_question)
print("\nBASELINE MODEL (no RAG):")
print(llm.invoke(internal_question).content)

### Step 7.5 &mdash; Answers with sources attached

<div class='concept-box'>
Now the grounded version &mdash; and this time every answer arrives <strong>with its receipts</strong>.
An answer a manager cannot audit is an answer a manager cannot sign off. Citation is not decoration;
it is the difference between "the AI said so" and "page 1 of the returns policy says so."
</div>


In [ ]:
def ask_with_sources(question: str, k: int = 4) -> None:
    """Answer from the document index, and show which files the evidence came from."""
    # search the store directly so the caller can control k per question
    relevant = doc_vectorstore.similarity_search(question, k=k)
    context  = "\n\n".join(c.page_content for c in relevant)
    answer   = llm.invoke(rag_prompt.format(context=context, question=question)).content

    print(f"Q: {question}")
    print(f"\nA: {answer}")
    print("\n   Sources consulted:")
    seen = set()
    for c in relevant:
        src  = os.path.basename(c.metadata.get("source", "unknown"))
        page = c.metadata.get("page")
        tag  = f"{src} (page {page + 1})" if page is not None else src
        if tag not in seen:
            seen.add(tag)
            print(f"     - {tag}")
    print("=" * 72)


ask_with_sources(internal_question)

In [ ]:
# One question per format, to prove all three files are genuinely searchable.

ask_with_sources("At what backlog size does the POS_SYNC_LAG alert become critical, "
                 "and which team gets paged?")                       # -> the Word table

ask_with_sources("What is the flat handling fee charged to a seller per unit when a return "
                 "is the seller's fault, and what is the dispute window?")   # -> the PDF

ask_with_sources("What is the one thing the runbook says you must NEVER do with "
                 "queued POS transactions?")                          # -> the Word doc

<div class='takeaway'>
<strong>Note what just happened with the Word table.</strong> The alert thresholds live in a
<em>table</em> inside the .docx. The loader flattened that grid into a line of text, and here the model
read it correctly.<br><br>
<strong>Hold that word "here".</strong> It got this one right with four chunks of context to work from.
In Part 9 we ask a different question of the same table and watch it go confidently wrong &mdash; twice.
<strong>Ingestion quality is the single most under-budgeted line item</strong> in enterprise RAG projects,
and flattened tables are where the bill arrives.
</div>


<hr class='section-divider'>

## Part 8 &mdash; The Refusal Test, Against Our Own Documents

<div class='concept-box'>
We ran this test on the web index. Run it again here, because <strong>this is the index that will
answer employees' questions</strong>. Three questions that sound exactly like our corpus &mdash;
HR-shaped, runbook-shaped, finance-shaped &mdash; and are answered nowhere in it.
</div>


In [ ]:
trick_internal = [
    "What is the on-call compensation rate per shift according to the handbook?",
    "What does runbook RB-104 say about payment-gateway certificate rotation?",
    "How many days of parental leave does the engineering handbook grant?",
    "What is the current share price of the company?",
]

print("REFUSAL TEST on internal documents — no fabrication allowed")
print("=" * 72)
for q in trick_internal:
    relevant = doc_retriever.invoke(q)
    context  = "\n\n".join(c.page_content for c in relevant)
    print(f"\nQ: {q}")
    print(f"A: {llm.invoke(rag_prompt.format(context=context, question=q)).content}")
    print("-" * 72)

<div class='warning-box'>
<strong>Note that RB-104 does not exist.</strong> RB-101 and RB-102 do. A system that answers a question
about a fabricated runbook number has not just made an error &mdash; it has invented an operational
procedure, and someone on-call at 2 a.m. may follow it. This is the test to put in your team's CI, not
in a one-off demo.
</div>


<hr class='section-divider'>

## Part 9 &mdash; Chunk Size, and the Quietest Failure in RAG

<div class='concept-box'>
Chunk size is the knob teams tune last and should tune first. Too small and a fact gets separated from
the context that makes it meaningful; too large and the retrieved chunk is mostly noise.<br><br>
We ask one question whose answer lives <strong>only inside the Word table</strong>, at three chunk
sizes. Read the answers carefully against the table in Step 7.2 before you accept any of them.
</div>

<div class='warning-box'>
<strong>Facilitator note:</strong> do not skip ahead to the explanation. Let the room read the three
answers first and decide which one they would ship.
</div>


In [ ]:
# A fact that lives ONLY inside the Word table — so we can watch the table
# survive or break at different chunk sizes.
test_query = ("Which team is paged when INVENTORY_FEED_STALE fires, "
              "and what is its critical threshold?")

for size in [200, 500, 1000]:
    splitter_n = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=int(size * 0.15))
    chunks_n   = splitter_n.split_documents(all_docs)
    store_n    = FAISS.from_documents(chunks_n, embedding)
    top        = store_n.similarity_search(test_query, k=1)[0]

    # Did the table's header row survive inside the retrieved chunk?
    flat       = " ".join(top.page_content.split())
    has_header = "Alert Warning Critical Paging team" in flat
    answer     = llm.invoke(
        rag_prompt.format(context=top.page_content, question=test_query)
    ).content

    print(f"CHUNK SIZE {size:>4} | {len(chunks_n):>3} chunks | top match {len(top.page_content)} chars")
    print(f"  column headers retrieved? {'YES' if has_header else 'NO  <-- the table lost its header'}")
    print(f"  retrieved : {flat[:200]}...")
    print(f"  ANSWER    : {answer.strip()[:200]}")
    print("-" * 72)

<div class='concept-box'>
<strong>The ground truth, from the table in the runbook:</strong>
INVENTORY_FEED_STALE &mdash; warning <strong>45 minutes</strong>, critical <strong>90 minutes</strong>,
paged team <strong>Supply Chain Platform</strong>.
</div>

<div class='warning-box'>
<strong>Now score the three answers:</strong>
<ul style='margin:6px 0 0 18px;'>
<li><strong>200 &mdash; refused.</strong> The split landed <em>inside the table</em> and severed the
header row, so the chunk was bare numbers: <code>...45 minutes 90 minutes Supply Chain Platform</code>.
Forty-five minutes of <em>what</em>? The model could not tell, and refused.
<strong>This is the only honest answer of the three.</strong></li>
<li><strong>500 &mdash; wrong.</strong> Right team, but it reported the critical threshold as
<strong>45 minutes</strong>. That is the <em>warning</em> value. It read across the wrong column.</li>
<li><strong>1,000 &mdash; wrong.</strong> Right threshold (90 minutes), but it named
<strong>Store Systems Reliability</strong> &mdash; the team from the row above.
It read down the wrong row.</li>
</ul>
</div>

<div class='takeaway'>
<strong>This is the lesson, and it is not the one people expect.</strong><br><br>
A Word table is a <em>grid</em>. Every loader we have flattens it into <em>a line of text</em>. Once
flattened, nothing tells the model which number belongs to which column &mdash; so it guesses, fluently,
and in this case it guessed differently at every chunk size. <strong>Two of the three answers were
confidently wrong, and both would page the wrong team or set the wrong alert.</strong><br><br>
Note what would <em>not</em> have caught this: a bigger model, a better embedding, a stricter prompt.
And note what the refusal at size 200 tells you &mdash; <strong>a system that refuses more often is not
a worse system.</strong> It was the only configuration that did not mislead us.<br><br>
<strong>The two questions for a design review:</strong><br>
1. <em>"What happens to our tables and spreadsheets at ingest?"</em> If the answer is "they get flattened
into text," your numeric answers are unreliable and no downstream tuning will fix it. Structured data
belongs in a database and gets <strong>queried</strong>, not embedded &mdash; which is exactly where E09 goes.<br>
2. <em>"How do we know when it is wrong?"</em> Nothing in these outputs looked broken. Someone had to
already know the right answer to spot it.
</div>

<div class='takeaway'>
<strong>The point for a design review:</strong> the 200-character failure is not a model problem, an
embedding problem, or a prompt problem. <strong>The fact was destroyed at ingest time</strong>, before
any model saw it &mdash; and no downstream fix recovers it.<br><br>
There is no universally correct chunk size. It depends on how your documents are written &mdash;
prose tolerates small chunks, tables and structured layouts do not. That is why this is a
<strong>design decision</strong>, not a default to leave at whatever the tutorial used.
</div>


<hr class='section-divider'>

## Part 10 &mdash; The Failure RAG Cannot Fix

<div class='concept-box'>
Everything so far has worked. Grounded answers, honest refusals, cited sources. So we finish with the
failure that survives all of it &mdash; and it is the one most likely to bite a team in production,
precisely because nothing looks broken.
</div>

<div class='concept-box'>
Our corpus has one more file: <code>shipment_status_memo.txt</code>, issued <strong>10 August</strong> by
regional inbound logistics. It is a perfectly ordinary business document. We add it to the index and
ask a perfectly ordinary business question.
</div>


In [ ]:
# Add the logistics memo to our index — a normal document, ingested normally.
memo_docs   = TextLoader("../data/shipment_status_memo.txt", encoding="utf-8").load()
memo_chunks = doc_splitter.split_documents(memo_docs)
doc_vectorstore.add_documents(memo_chunks)

print(f"Added {len(memo_chunks)} chunk(s) from the shipment status memo.")
print("Index now covers: handbook (txt) + runbook (docx) + returns policy (pdf) + logistics memo (txt)")

In [ ]:
# k=2 here: we want the citation list tight enough to read at a glance.
ask_with_sources("When is shipment SHP-88121 expected to arrive at store 4479, "
                 "and is it on schedule?", k=2)

<div class='concept-box'>
A clean answer, from a real document, with a correct citation. Every guardrail we built held.
<br><br>
Now let us look at what the <strong>live shipment tracking system</strong> says right now.
</div>


In [ ]:
import json

with open("../data/shipments.json") as f:
    live = json.load(f)

record = next(s for s in live if s["shipment_id"] == "SHP-88121")

print("LIVE SYSTEM OF RECORD — inbound shipment tracking")
print("=" * 72)
for k, v in record.items():
    print(f"  {k:15} {v}")

<div class='warning-box'>
<strong>The memo said 11 August, on schedule. The live system says 18 August, delayed &mdash;
carrier weather hold, seven days late.</strong><br><br>
Read the failure carefully, because the interesting part is what did <em>not</em> go wrong:
<ul style='margin:6px 0 0 18px;'>
<li>Retrieval found the correct chunk.</li>
<li>The model answered strictly from context &mdash; exactly as instructed.</li>
<li>The citation was accurate. The document really does say 11 August.</li>
<li>No hallucination occurred. Not one guardrail failed.</li>
</ul>
<strong>And the answer was still wrong</strong>, in a way that sends a replenishment team to reset a
shelf for stock that will not arrive for another week.
</div>

<div class='takeaway'>
<strong>The rule to take out of this room:</strong><br><br>
<strong>A document is a photograph.</strong> RAG grounds you in what was true when someone last wrote it
down. Nobody re-indexed the memo when the carrier called &mdash; and no amount of better chunking,
better embeddings or a bigger model would have changed the answer by one day.<br><br>
<strong>STATIC knowledge</strong> &mdash; policies, runbooks, closed quarters, handbooks &mdash;
<strong>&rarr; RAG.</strong> Embed once, retrieve from the snapshot.<br>
<strong>LIVE knowledge</strong> &mdash; shipments, stock counts, prices, ticket status &mdash;
<strong>&rarr; the model must ask the system of record, at question time.</strong><br><br>
The question is never "is this data simple or complex?" It is
<strong>"is this data static or live?"</strong>
</div>


<hr class='section-divider'>

## Recap

<table class='compare-table'>
<tr><th>Part</th><th>What we did</th><th>Manager takeaway</th></tr>
<tr><td>3</td><td>Asked the raw model for an exact figure and a named person</td><td>Hallucination is fluent and confident, not garbled — you cannot spot it by tone</td></tr>
<tr><td>4</td><td>Loaded a URL, chunked it, embedded it, built a retriever</td><td>Four steps, no magic; retrieval is where most RAG bugs actually live</td></tr>
<tr><td>5</td><td>Re-asked the same question, grounded</td><td>Same model, same prompt — only the visible evidence changed</td></tr>
<tr><td>6</td><td>Tested retrieval accuracy AND refusal</td><td>A system that answers well but never refuses is more dangerous than none</td></tr>
<tr><td>7</td><td>Ingested .txt + .docx + .pdf into one index, with citations</td><td>Only the loader changes; ingestion quality is the under-budgeted line item</td></tr>
<tr><td>8</td><td>Asked about a runbook that does not exist</td><td>An invented procedure is worse than an invented fact — put this test in CI</td></tr>
<tr><td>9</td><td>Asked one table-only question at three chunk sizes</td><td>Flattened tables produce confident wrong answers; the refusal was the honest result</td></tr>
<tr><td>10</td><td>Answered correctly from a stale memo</td><td><strong>Static → RAG. Live → tools.</strong> RAG cannot fix a document that aged</td></tr>
</table>

<div class='concept-box'>
<strong>Glossary</strong>
<table class='compare-table'>
<tr><td><strong>Chunk</strong></td><td>A small, overlapping slice of a document sized to fit usefully in a prompt</td></tr>
<tr><td><strong>Embedding</strong></td><td>A vector of numbers (here, 1,536) representing a text's meaning; similar meaning &rarr; nearby vectors</td></tr>
<tr><td><strong>Vector store</strong></td><td>An index (here, FAISS) supporting fast nearest-neighbour search over embeddings</td></tr>
<tr><td><strong>Retriever</strong></td><td>The component that embeds a question and returns the top-k most similar chunks</td></tr>
<tr><td><strong>Top-k</strong></td><td>How many chunks retrieval returns per question (we used k=4)</td></tr>
<tr><td><strong>Grounding</strong></td><td>Constraining the model to answer only from supplied context, with refusal otherwise</td></tr>
<tr><td><strong>Hallucination</strong></td><td>A fluent, confident answer not supported by the source material</td></tr>
<tr><td><strong>Document loader</strong></td><td>The format-specific reader (.txt / .docx / .pdf / web) that produces text + metadata</td></tr>
</table>
</div>


<div class='topic-header'>
<strong>&#128279; The two gaps we leave</strong><br><br>
<strong>Gap 1 &mdash; structured data (next, E09).</strong> Our assistant answers from
<em>documents</em> &mdash; policies, runbooks, prose. But ask it <em>"how many orders did the
South-Central region lose to stockouts last week?"</em> and there is no paragraph to retrieve.
That answer lives in <strong>rows and columns</strong>, and it has to be computed, not quoted.
Embedding a database does not work. E09 closes that gap.<br><br>
<strong>Gap 2 &mdash; live data (E10).</strong> The one we just watched break. The shipment memo aged,
and retrieval faithfully served us a week-old fact. To get the real answer, the model has to
<strong>call the tracking system itself</strong>, at the moment the question is asked.
That is tool calling and MCP &mdash; and it is where E10 begins, with this exact shipment.
</div>
